# Named Entity Recognition (NER) Pipeline
# Code đầy đủ và data tại: https://github.com/duyhoang17930/Named-Entity-Recognition
## Complete Pipeline: Crawl → Preprocess → label → Encode →  Train → Evaluate → Predict

This notebook contains all steps for building an NER system:
1. **Day1 - Crawl**: Data collection from Google News
2. **Day2 - Preprocess**: Text cleaning and preprocessing
3. **Day3 - Encode**: NER labeling with spaCy
5. **Day4 - Relabeling**: Apply custom entity mappings for improved accuracy
6. **Day5 - Train**: Model training with DistilBERT
7. **Day6 - Results**: Evaluation, visualization, and prediction

# Step 3: NER Encoding (Day3)

Using spaCy for NER labeling:
- Load spaCy model
- Extract entities (PERSON, ORG, GPE, DATE, etc.)
- Apply rule-based corrections
- Encode tokens and labels

In [93]:
import sys
import subprocess

def run_cmd(cmd):
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Command failed")

# Cài spaCy vào đúng Python interpreter của notebook
run_cmd([sys.executable, "-m", "pip", "install", "-q", "spacy"])

# Tải model en_core_web_sm
run_cmd([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

Running: c:\Users\pc\AppData\Local\Programs\Python\Python313\python.exe -m pip install -q spacy

Running: c:\Users\pc\AppData\Local\Programs\Python\Python313\python.exe -m spacy download en_core_web_sm
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
âœ” Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



In [9]:
import pandas as pd
import spacy
import numpy as np
import pickle
import re

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Read dataset
df = pd.read_csv("clean_news.csv")
sentences = df["clean_sentence"].dropna().tolist()

print(f"Total sentences: {len(sentences)}")

Total sentences: 3167


In [10]:
# Metrics
def compute_metrics(p):
    from seqeval.metrics import classification_report
    
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []
    
    for pred, label in zip(predictions, labels):
        temp_pred = []
        temp_label = []
        for pred_id, label_id in zip(pred, label):
            if label_id != -100:
                temp_pred.append(id2label[int(pred_id)])
                temp_label.append(id2label[int(label_id)])
        if temp_pred:
            true_predictions.append(temp_pred)
            true_labels.append(temp_label)

    report = classification_report(true_labels, true_predictions, digits=4)
    print("\n" + report)
    return {}

In [ ]:
# NER labeling process
dataset = []

for sentence in sentences:
    doc = nlp(sentence)

    tokens = [token.text for token in doc]
    labels = ["O"] * len(tokens)

    # Extract entities from spaCy
    for ent in doc.ents:
        start = ent.start
        end = ent.end
        label = ent.label_

        labels[start] = "B-" + label
        for i in range(start + 1, end):
            labels[i] = "I-" + label

    # Apply corrections
    labels = correct_entities(tokens, labels)

    dataset.append((tokens, labels))

# Show example
print("Example tokens + labels:")
for t, l in zip(dataset[0][0], dataset[0][1]):
    print(f"  {t:<15} {l}")

In [14]:
# Create vocabulary
token2id = {}
id2token = {}

idx = 1
for tokens, labels in dataset:
    for token in tokens:
        if token not in token2id:
            token2id[token] = idx
            id2token[idx] = token
            idx += 1

# Encode labels
label_set = set()
for tokens, labels in dataset:
    label_set.update(labels)

label2id = {label: i for i, label in enumerate(sorted(label_set))}
id2label = {i: label for label, i in label2id.items()}

print("Label set:", label2id)
print(f"Vocabulary size: {len(token2id)}")

Label set: {}
Vocabulary size: 0


In [12]:
# Encode dataset
encoded_dataset = []

for tokens, labels in dataset:
    token_ids = [token2id[token] for token in tokens]
    label_ids = [label2id[label] for label in labels]

    encoded_dataset.append({
        "tokens": token_ids,
        "labels": label_ids
    })

# Padding
MAX_LEN = 50

def pad(seq, max_len, pad_value=0):
    return seq[:max_len] + [pad_value] * (max_len - len(seq))

X = []
y = []

for item in encoded_dataset:
    X.append(pad(item["tokens"], MAX_LEN))
    y.append(pad(item["labels"], MAX_LEN))

X = np.array(X)
y = np.array(y)

print(f"Dataset shape: X={X.shape}, y={y.shape}")

Dataset shape: X=(0,), y=(0,)


In [15]:
# Save encoded dataset
with open("ner_dataset.pkl", "wb") as f:
    pickle.dump({
        "X": X,
        "y": y,
        "token2id": token2id,
        "label2id": label2id,
        "id2label": id2label
    }, f)

print("Saved dataset to ner_dataset.pkl")

Saved dataset to ner_dataset.pkl


---

# Step 4: Auto-Labeling with custom mapping (Day4)

